# 从零实现 BPE Tokenizer

> **前情回顾**：上一节我们试过字符级 tokenizer 和词级 tokenizer。字符级不会崩，但太碎；词级看起来清楚，但遇到没见过的词就麻烦。
>
> **本 Part 目标**：你会亲手写出一个 mini BPE tokenizer，并看懂它为什么能把字符一步步合成子词。

## 学习地图

先别急着看代码。BPE 这节最重要的不是背公式，而是抓住 4 个问题：

1. **为什么要从字符开始？** 因为字符最稳，至少不会轻易遇到 OOV。
2. **为什么要合并高频 pair？** 因为高频组合最值得压缩成一个 token。
3. **为什么 merge rules 有顺序？** 因为 encode 新文本时，要按训练时的顺序重放这些合并。
4. **为什么词表大小可控？** 因为你做几次 merge，就新增几个 token。

一句话记住 BPE：**反复找最常出现的相邻 token 对，把它们合并成一个新 token。**

你可以把它想成搭积木：一开始只有小积木（字符），BPE 每次把最常连在一起的两块粘起来，慢慢长出 `th`、`the `、`ing` 这样的子词。


## 1. 为什么要 BPE

我们先把问题说清楚：**Tokenizer 到底在折中什么？**

| 方案 | 好处 | 麻烦 |
|:---|:---|:---|
| 字符级 | 几乎不会遇到生词 | 序列太长，模型看得累 |
| 词级 | token 更像人类理解的词 | 没见过的词容易 OOV |
| BPE 子词级 | 常见词能合并，生词能拆开 | 需要先训练 merge rules |

所以 BPE 的目标很朴素：

**常见的东西合大一点，不常见的东西保留小一点。**

比如 `the` 很常见，就值得变成一个 token；但一个没见过的新词，也可以退回到字符或子词，不至于直接崩掉。

下面我们从一个很小的语料开始。语料小，反而更适合观察：每一次合并都能看清楚。


**准备语料：还是用小例子，方便你盯住每一步**


In [1]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print("我们的语料：")
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

我们的语料：
  [0] the cat sat on the mat
  [1] the dog sat on the log
  [2] the cat and the dog
  [3] i love my cat
  [4] i love my dog
  [5] the cat is cute
  [6] the dog is happy
  [7] the mat is soft
  [8] the log is hard
  [9] cats and dogs are friends


## 2. 训练 BPE

### 起点：字符

BPE 的起点是**字符级**：先把文本拆成字符，每个字符都是一个 token。

为什么从字符开始？因为字符虽然碎，但很稳。只要字符在词表里，新词就还能被拆开表示。

还有一个细节：空格怎么处理？

真实 GPT tokenizer 会对空格做更精细的处理。为了让你先看懂主流程，我们这个 mini 版先把空格也当普通字符。这样你会清楚看到：BPE 不只会合并字母，也可能把 `e + 空格` 这样的组合合起来。


In [2]:
# 第一步：把所有句子拆成字符列表
# 注意：空格 ' ' 就是一个普通字符

def text_to_tokens(text):
    """把一段文本拆成字符级 token 列表"""
    return list(text)

# 看第一条句子
sentence = corpus[0]
chars = text_to_tokens(sentence)

print(f"原文:   '{sentence}'")
print(f"拆成字符: {chars}")
print(f"共 {len(chars)} 个 token")

# 把整条语料的每条句子都拆成字符列表
corpus_tokens = [text_to_tokens(s) for s in corpus]
print(f"\n语料中每条句子的字符数: {[len(t) for t in corpus_tokens]}")

原文:   'the cat sat on the mat'
拆成字符: ['t', 'h', 'e', ' ', 'c', 'a', 't', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e', ' ', 'm', 'a', 't']
共 22 个 token

语料中每条句子的字符数: [22, 22, 19, 13, 13, 15, 16, 15, 15, 25]


### 统计 pair

现在我们要问一个很机械的问题：**哪两个相邻 token 最常一起出现？**

比如这个序列：

```
['t', 'h', 'e', ' ', 'c', 'a', 't']
  ↑──↑
 ('t','h'): 1次
      ↑──↑
     ('h','e'): 1次
          ↑──↑
         ('e',' '): 1次
              ↑──↑
             (' ','c'): 1次
                  ↑──↑
                 ('c','a'): 1次
                      ↑──↑
                     ('a','t'): 1次
```

单个句子里看不出太多规律，所以我们要统计**整个语料**里的所有相邻 pair。谁出现最多，谁就最值得先合并。


In [3]:
from collections import defaultdict

def count_pairs(token_lists):
    """
    统计整个语料中所有相邻 token pair 的出现频率
    
    参数:
        token_lists: List[List[str]]，每条句子是一个字符列表
    返回:
        dict: {(token_a, token_b): 出现次数}
    """
    pairs = defaultdict(int)
    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += 1
    return dict(pairs)

# 统计初始状态（全字符级）的 pair 频率
initial_pairs = count_pairs(corpus_tokens)

print("=== 初始状态下所有相邻 pair 的频率 ===")
# 按频率从高到低排序打印
for pair, count in sorted(initial_pairs.items(), key=lambda x: -x[1]):
    print(f"  {pair}: {count} 次")

print(f"\n最高频 pair: {max(initial_pairs, key=initial_pairs.get)}")
print(f"出现了 {initial_pairs[max(initial_pairs, key=initial_pairs.get)]} 次")

=== 初始状态下所有相邻 pair 的频率 ===
  ('e', ' '): 13 次
  ('t', 'h'): 10 次
  ('h', 'e'): 10 次
  ('a', 't'): 9 次
  ('o', 'g'): 7 次
  ('t', ' '): 6 次
  ('s', ' '): 6 次
  (' ', 'c'): 5 次
  ('c', 'a'): 5 次
  (' ', 'd'): 5 次
  ('d', 'o'): 5 次
  (' ', 'm'): 4 次
  (' ', 'l'): 4 次
  ('l', 'o'): 4 次
  (' ', 'i'): 4 次
  ('i', 's'): 4 次
  (' ', 's'): 3 次
  (' ', 't'): 3 次
  ('g', ' '): 3 次
  (' ', 'a'): 3 次
  ('n', 'd'): 3 次
  ('s', 'a'): 2 次
  (' ', 'o'): 2 次
  ('o', 'n'): 2 次
  ('n', ' '): 2 次
  ('m', 'a'): 2 次
  ('a', 'n'): 2 次
  ('d', ' '): 2 次
  ('i', ' '): 2 次
  ('o', 'v'): 2 次
  ('v', 'e'): 2 次
  ('m', 'y'): 2 次
  ('y', ' '): 2 次
  (' ', 'h'): 2 次
  ('h', 'a'): 2 次
  ('a', 'r'): 2 次
  ('c', 'u'): 1 次
  ('u', 't'): 1 次
  ('t', 'e'): 1 次
  ('a', 'p'): 1 次
  ('p', 'p'): 1 次
  ('p', 'y'): 1 次
  ('s', 'o'): 1 次
  ('o', 'f'): 1 次
  ('f', 't'): 1 次
  ('r', 'd'): 1 次
  ('t', 's'): 1 次
  ('g', 's'): 1 次
  ('r', 'e'): 1 次
  (' ', 'f'): 1 次
  ('f', 'r'): 1 次
  ('r', 'i'): 1 次
  ('i', 'e'): 1 次
  ('e', 'n'): 1 

### 合并高频 pair

找到最高频 pair 后，BPE 会把语料里**所有这个 pair** 都合并掉。

```
合并前: ['t', 'h', 'e', ' ', 'c', 'a', 't']
假设要合并: ('t', 'h')
合并后: ['th', 'e', ' ', 'c', 'a', 't']
            ↑  't' 和 'h' 变成一个 token 'th'
```

这一步像什么？像把常一起出现的两个小零件焊成一个大零件。下次再统计时，`th` 就会作为一个整体参与统计。


In [4]:
def merge_pair(token_lists, pair_to_merge):
    """
    把语料中所有出现的指定 pair 合并成一个 token
    
    参数:
        token_lists: List[List[str]]，当前语料的 token 表示
        pair_to_merge: (str, str)，要合并的 pair
    返回:
        合并后的新 token_lists
    """
    a, b = pair_to_merge
    new_token = a + b  # 新 token 就是两个字符拼接
    
    new_token_lists = []
    for tokens in token_lists:
        new_tokens = []
        i = 0
        while i < len(tokens):
            # 如果当前位置和下一个位置正好是这个 pair，合并！
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(new_token)
                i += 2  # 跳两步
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_token_lists.append(new_tokens)
    
    return new_token_lists, new_token

# 演示：做一次合并
best_pair = max(initial_pairs, key=initial_pairs.get)
print(f"要合并的 pair: {best_pair}")

merged_tokens, new_token = merge_pair(corpus_tokens, best_pair)
print(f"新 token: '{new_token}'")
print(f"\n合并后每条句子的 token 序列：")
for i, tokens in enumerate(merged_tokens):
    print(f"  [{i}] {tokens}")

要合并的 pair: ('e', ' ')
新 token: 'e '

合并后每条句子的 token 序列：
  [0] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e ', 'm', 'a', 't']
  [1] ['t', 'h', 'e ', 'd', 'o', 'g', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e ', 'l', 'o', 'g']
  [2] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 'a', 'n', 'd', ' ', 't', 'h', 'e ', 'd', 'o', 'g']
  [3] ['i', ' ', 'l', 'o', 'v', 'e ', 'm', 'y', ' ', 'c', 'a', 't']
  [4] ['i', ' ', 'l', 'o', 'v', 'e ', 'm', 'y', ' ', 'd', 'o', 'g']
  [5] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 'i', 's', ' ', 'c', 'u', 't', 'e']
  [6] ['t', 'h', 'e ', 'd', 'o', 'g', ' ', 'i', 's', ' ', 'h', 'a', 'p', 'p', 'y']
  [7] ['t', 'h', 'e ', 'm', 'a', 't', ' ', 'i', 's', ' ', 's', 'o', 'f', 't']
  [8] ['t', 'h', 'e ', 'l', 'o', 'g', ' ', 'i', 's', ' ', 'h', 'a', 'r', 'd']
  [9] ['c', 'a', 't', 's', ' ', 'a', 'n', 'd', ' ', 'd', 'o', 'g', 's', ' ', 'a', 'r', 'e ', 'f', 'r', 'i', 'e', 'n', 'd', 's']


**一次合并之后，世界变了**

注意：合并不是只改了词表，它也改了后续统计的对象。

BPE 的训练循环其实只有 4 步：

1. 统计 pair 频率
2. 找最高频 pair
3. 合并这个 pair
4. 记录这条合并规则，也就是 **merge rule**

然后回到第 1 步。

每次合并都会制造新的相邻关系。比如先得到 `th`，后面就可能继续得到 `the `。所以 BPE 不是一下子认出一个词，而是一步步长出来。


## 3. 训练日志

现在我们把上面的循环真正跑起来。

这里有一个最重要的参数：`num_merges`。

- `num_merges=10`：新增 10 个 token，词表小，文本会切得更碎
- `num_merges=50`：新增 50 个 token，词表更大，常见片段会更完整
- 大模型 tokenizer：通常会做很多很多次 merge，词表能到几万甚至更多 token

我们先设 `num_merges=15`。不要急着看最终结果，重点看日志里每一步发生了什么。


In [5]:
class BPETokenizer:
    """
    BPE Tokenizer 完整实现
    
    三个核心功能：
    1. train()  — 从语料学习 merge rules
    2. encode() — 文本 → token ID 列表
    3. decode() — token ID 列表 → 文本
    """
    
    def __init__(self):
        # merge_rules: 按顺序记录每次合并的 pair
        # 这是 BPE 最核心的数据结构，encode 时按这个顺序贪心应用
        self.merge_rules = []
        # 最终词表：初始字符 + 每一步 merge 学到的新 token
        self.vocab = set()
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts, num_merges=15, verbose=True):
        """
        BPE 训练
        
        参数:
            texts: 语料列表
            num_merges: 合并次数
            verbose: 是否打印每一步
        """
        # Step 0: 初始化为字符级
        token_lists = [list(text) for text in texts]
        base_vocab = set(c for tokens in token_lists for c in tokens)
        learned_vocab = set(base_vocab)
        
        if verbose:
            print(f"{'='*60}")
            print(f"BPE 训练开始！初始状态: 每条句子按字符拆分")
            print(f"{'='*60}")
            print(f"初始字符集大小: {len(set(c for t in token_lists for c in t))}")
            print()
        
        for step in range(num_merges):
            # Step 1: 统计所有 pair 的频率
            pairs = defaultdict(int)
            for tokens in token_lists:
                for i in range(len(tokens) - 1):
                    pairs[(tokens[i], tokens[i + 1])] += 1
            
            if not pairs:
                break
            
            # Step 2: 找最高频 pair
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]
            
            # Step 3: 记录 merge rule
            self.merge_rules.append(best_pair)
            
            # Step 4: 合并
            a, b = best_pair
            new_token = a + b
            
            new_token_lists = []
            for tokens in token_lists:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                        new_tokens.append(new_token)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                new_token_lists.append(new_tokens)
            
            token_lists = new_token_lists
            
            learned_vocab.add(new_token)
            
            if verbose:
                print(f"Step {step+1:2d}: merge {best_pair} → '{new_token}'  (出现 {best_count} 次) | 当前词表大小: {len(learned_vocab)}")
        
        # 建立最终词表
        # 注意：词表不能只保留最终语料里还出现的 token。
        # encode 遇到没见过的组合时，需要能退回到初始字符。
        self.vocab = learned_vocab
        sorted_vocab = sorted(list(self.vocab))
        self.stoi = {t: i for i, t in enumerate(sorted_vocab)}
        self.itos = {i: t for i, t in enumerate(sorted_vocab)}
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"训练完成！")
            print(f"  - 合并次数: {len(self.merge_rules)}")
            print(f"  - 最终词表大小: {len(self.vocab)} 个 token")
            print(f"  - Merge rules: {self.merge_rules}")
            print(f"{'='*60}")
        
        return token_lists


In [6]:
# 训练 BPE tokenizer！设 15 步合并，每一步都打印
bpe = BPETokenizer()
final_tokens = bpe.train(corpus, num_merges=15, verbose=True)

BPE 训练开始！初始状态: 每条句子按字符拆分
初始字符集大小: 20

Step  1: merge ('e', ' ') → 'e '  (出现 13 次) | 当前词表大小: 21
Step  2: merge ('t', 'h') → 'th'  (出现 10 次) | 当前词表大小: 22
Step  3: merge ('th', 'e ') → 'the '  (出现 10 次) | 当前词表大小: 23
Step  4: merge ('a', 't') → 'at'  (出现 9 次) | 当前词表大小: 24
Step  5: merge ('o', 'g') → 'og'  (出现 7 次) | 当前词表大小: 25
Step  6: merge ('at', ' ') → 'at '  (出现 6 次) | 当前词表大小: 26
Step  7: merge ('s', ' ') → 's '  (出现 6 次) | 当前词表大小: 27
Step  8: merge ('d', 'og') → 'dog'  (出现 5 次) | 当前词表大小: 28
Step  9: merge ('i', 's ') → 'is '  (出现 4 次) | 当前词表大小: 29
Step 10: merge ('the ', 'c') → 'the c'  (出现 3 次) | 当前词表大小: 30
Step 11: merge ('the c', 'at ') → 'the cat '  (出现 3 次) | 当前词表大小: 31
Step 12: merge (' ', 'the ') → ' the '  (出现 3 次) | 当前词表大小: 32
Step 13: merge ('n', 'd') → 'nd'  (出现 3 次) | 当前词表大小: 33
Step 14: merge ('s', 'at ') → 'sat '  (出现 2 次) | 当前词表大小: 34
Step 15: merge ('sat ', 'o') → 'sat o'  (出现 2 次) | 当前词表大小: 35

训练完成！
  - 合并次数: 15
  - 最终词表大小: 35 个 token
  - Merge rules: [('e', ' '), ('

**Keypoint 4：看日志时，只看“谁先长出来”**

回头看上面的训练日志，你会发现它很像一个生长过程：

```
Step 1: ('e', ' ')   → 'e '     ← e 后面接空格很常见
Step 2: ('t', 'h')   → 'th'     ← the 出现多，所以 th 先出现
Step 3: ('th', 'e ') → 'the '   ← th 和 e 空格继续合成 the 空格
Step 4: ('a', 't')   → 'at'     ← cat / sat / mat 让 at 很常见
Step 5: ('o', 'g')   → 'og'     ← dog / log 里的 og 也被合并
```

**关键观察**：BPE 不是“认识英文单词”的程序。它只是很诚实地做统计：谁经常挨在一起，谁就先合并。

这也解释了为什么词表大小可控：你说“只做 15 次合并”，它就只新增 15 个 token。


## 4. Encode

### 按顺序合并

训练结束后，我们已经有了一串 merge rules。现在来了新文本，怎么编码？

**注意**：默认新文本中的所有字符已存在于词表中。

流程很简单：

1. 先把新文本拆成字符
2. 按训练时的 merge rules 顺序，一条一条尝试合并
3. 最后把剩下的 token 查成 ID

```
"the cat"
  → ['t','h','e',' ','c','a','t']
  → 应用 rule 1: ('e',' ') → 'e '
  → ['t','h','e ','c','a','t']
  → 应用 rule 2: ('t','h') → 'th'
  → ['th','e ','c','a','t']
  → 应用 rule 3: ('th','e ') → 'the '
  → ['the ','c','a','t']
  → 应用 rule 4: ('a','t') → 'at'
  → ['the ','c','at']
```

这里最容易踩坑的是：**顺序不能乱。**

因为先合并什么，会影响后面还能不能合并。encode 不是重新统计频率，而是按训练时学到的规则照着做。


In [7]:
# 给 BPETokenizer 加上 encode 方法
def bpe_encode(self, text):
    """
    BPE 编码：文本 → token ID 列表
    
    核心逻辑：按训练时的 merge rules 顺序，贪心地合并
    """
    # Step 1: 拆成字符
    tokens = list(text)
    
    # Step 2: 依次应用每条 merge rule
    for (a, b) in self.merge_rules:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    # Step 3: 每个 token 去词表查 ID, 没见过的单词不会在Step2中进行merge, 因此不会出现OOV.
    ids = []
    for token in tokens:
        if token in self.stoi:
            ids.append(self.stoi[token])
    return ids

# 把方法挂上去
BPETokenizer.encode = bpe_encode

print("encode 方法已添加！")

encode 方法已添加！


In [8]:
# 测试 encode
text = "the cat"
ids = bpe.encode(text)
print(f"原文: '{text}'")
print(f"Token IDs: {ids}")
print(f"Token 数量: {len(ids)}")

# 看看每个 ID 对应什么 token
print(f"\n逐个解释:")
for i, tid in enumerate(ids):
    print(f"  ID {tid} → '{bpe.itos[tid]}'")

# 对比字符级
print(f"\n对比: 原文字符数 = {len(text)}，BPE token 数 = {len(ids)}，压缩比 = {len(text)/len(ids):.1f}x")

原文: 'the cat'
Token IDs: [30, 3]
Token 数量: 2

逐个解释:
  ID 30 → 'the c'
  ID 3 → 'at'

对比: 原文字符数 = 7，BPE token 数 = 2，压缩比 = 3.5x


**Keypoint 6：decode 只是查表再拼回去**

decode 比 encode 简单很多。

它不需要统计，也不需要合并。它只做一件事：把 token ID 查回字符串，然后拼起来。

你可以把 encode 想成“切开并编号”，decode 想成“按编号找回碎片，再粘回原文”。


In [9]:
def bpe_decode(self, ids):
    """
    BPE 解码：token ID 列表 → 文本
    就是查表 + 拼接，没有任何其他操作
    """
    return "".join([self.itos[i] for i in ids])

BPETokenizer.decode = bpe_decode

# 测试完整编解码
for test_text in ["the cat", "my dog is happy", "i love cats"]:
    ids = bpe.encode(test_text)
    recovered = bpe.decode(ids)
    status = "✅" if test_text == recovered else "❌"
    print(f"{status} '{test_text}' → {ids} → '{recovered}'")

✅ 'the cat' → [30, 3] → 'the cat'
✅ 'my dog is happy' → [16, 34, 0, 7, 0, 14, 12, 2, 21, 21, 34] → 'my dog is happy'
✅ 'i love cats' → [13, 0, 15, 19, 33, 9, 5, 3, 23] → 'i love cats'


**Keypoint 7：BPE 为什么不怕大多数生词？**

看 encode 里的最后一步：

```python
# Step 3: 每个 token 去词表查 ID
ids = []
for token in tokens:
    if token in self.stoi:
        ids.append(self.stoi[token])
return ids

```

你会发现，这里**根本没有写处理未知词的 `else` 分支**！这就是 BPE 算法的精妙之处，也是它比传统词级 tokenizer 更稳的原因。

因为我们的编码是从**单字符**开始的（Step 1），而 Step 2 仅仅是照着训练好的 `merge_rules` 进行合并。如果遇到没见过的新词（例如训练时只见过 "play" 和 "ing"，没见过 "playing"），它在 Step 2 最多只会合并到子词级别，自然而然地输出 `['play', 'ing']`，**绝对不会合成一个不在词表里的生词**。所以到了 Step 3，所有的 token 都是 100% 保证在词表里的。

**不过要注意前置条件：**
我们这个 mini 版能跑通，是因为有一个大前提——**“默认新文本中的所有字符已存在于词表中”**。如果新文本里出现了一个训练语料从没见过的全新字符（比如罕见的 emoji `👽`），它在 Step 3 依然会找不到 ID。

这也是为什么工业级 tokenizer 通常不从字符开始，而是采用 Byte-Level BPE（字节级 BPE）。

因为 256 个基础 byte 就可以组合出世界上任何文字与符号，这种设计从物理层面上彻底根绝了 OOV（Out-Of-Vocabulary）问题。

同样是面对刚才那个从未见过的 '👽'，Byte-Level BPE 不会将其视为一个无法识别的新字符，而是先转化为 UTF-8 编码下的 4 个字节：`[240, 159, 145, 189]`。哪怕系统从未学过要把它们合并成外星人 emoji，在 Step 3 查表时，它依然能在 0-255 的基础词表中稳稳查到这 4 个独立字节的 ID。这样一来，代码就再也不会因为“遇到了没见过的


In [10]:
# 演示：编码一个语料中不存在的句子
unseen = "a cat runs fast"
print(f"测试生僻句: '{unseen}'")
print(f"runs fast 没有出现在训练语料中，但这些字符训练时都见过。")

ids = bpe.encode(unseen)
recovered = bpe.decode(ids)

print(f"\nToken IDs: {ids}")
print(f"解码回来: '{recovered}'")
print(f"状态: {'✅ 没崩！' if unseen == recovered else '⚠️'}")

# 看看 BPE 把它拆成了什么
print(f"\n逐个 token:")
for tid in ids:
    print(f"  '{bpe.itos[tid]}'", end="")
print()

测试生僻句: 'a cat runs fast'
runs fast 没有出现在训练语料中，但这些字符训练时都见过。

Token IDs: [2, 0, 5, 4, 22, 32, 17, 24, 10, 2, 23, 27]
解码回来: 'a cat runs fast'
状态: ✅ 没崩！

逐个 token:
  'a'  ' '  'c'  'at '  'r'  'u'  'n'  's '  'f'  'a'  's'  't'


## 5. Merge 次数

merge 次数越多，词表越大，token 越“整”。

但这不是越大越好。词表太小，序列会很长；词表太大，很多 token 很少出现，学习也不划算。

我们用一个小实验看这件事：同一句话，在不同 merge 次数下会被切成什么样。


In [11]:
# 用不同 merge 次数训练多个 tokenizer
for n_merges in [5, 15, 30]:
    t = BPETokenizer()
    t.train(corpus, num_merges=n_merges, verbose=False)
    
    test = "the cat sat on the mat"
    ids = t.encode(test)
    
    print(f"merge={n_merges:2d} | 词表={len(t.vocab):2d} | token数={len(ids):2d} | tokens: {[t.itos[i] for i in ids]}")

print()
print("观察：merge 次数越多，常见片段越可能作为整体保留，token 数越少。")

merge= 5 | 词表=25 | token数=13 | tokens: ['the ', 'c', 'at', ' ', 's', 'at', ' ', 'o', 'n', ' ', 'the ', 'm', 'at']
merge=15 | 词表=35 | token数= 6 | tokens: ['the cat ', 'sat o', 'n', ' the ', 'm', 'at']
merge=30 | 词表=50 | token数= 4 | tokens: ['the cat ', 'sat on the ', 'm', 'at']

观察：merge 次数越多，常见片段越可能作为整体保留，token 数越少。


## 6. 真实 BPE

### 分词的影响

BPE 很实用，但它不是完美答案。它会把文本压缩成 token，而压缩就可能改变模型看到的结构。

**例子 1：数字可能被藏起来**

数字 `327` 可能被整体编码成一个 token，而不是 `3`、`2`、`7` 三个字符。这样模型就不一定天然理解“百位、十位、个位”。

所以它做 `327 + 1` 时，不一定像你一样从个位开始进位。对模型来说，这可能只是一个 token 和另一个 token 的关系。

**例子 2：代码里的空格很敏感**

Python 的缩进有语法意义。4 个空格如果被切得不稳定，模型理解代码结构就会更难。

这也是为什么面向代码的模型，会特别优化空格、换行和缩进相关的 token。

**例子 3：为什么不直接用字符？**

直接用字符或 byte 当然更“无损”，数字和空格结构也更清楚。

但代价是序列会变长很多。Transformer 看长序列很贵，因为注意力计算通常跟序列长度的平方有关。

所以现在的主流选择还是：用 BPE 或类似方法，在“信息完整”和“计算成本”之间折中。


**Keypoint 9：工业版 BPE 比我们的 mini 版更稳、更快**

我们的实现是教学版，故意写得直白。真实 tokenizer 会做更多工程优化：

| 我们的版本 | 工业版 |
|:---|:---|
| 从字符开始 | 从 **bytes** 开始，能覆盖所有 Unicode |
| 空格当普通字符 | 会专门处理词首空格、换行、缩进 |
| 全量统计频率 | 用更高效的数据结构加速 |
| 小语料 | 海量语料 |
| 纯 Python | 常用 C++ / Rust 实现 |

“从 bytes 开始”是什么意思？

字符级起点看起来已经很稳了，但 Unicode 世界太大了。比如：

```text
"你" 这个字符，用 UTF-8 存储时其实是 3 个 byte：E4 BD A0
"😊" 这个 emoji，用 UTF-8 存储时是 4 个 byte：F0 9F 98 8A
```

如果 tokenizer 从字符开始，它需要提前认识各种中文、emoji、特殊符号。
但如果从 byte 开始，起点永远只有 256 种可能：`0` 到 `255`。
这样无论你输入中文、日文、emoji，还是奇怪符号，至少都能先拆成 bytes，不会直接 OOV 崩掉。

也就是说：

```text
字符级：先问“这个字符我认识吗？”
byte 级：任何字符都先拆成 byte，所以一定有入口
```

但核心思想没变：**统计 pair → 合并最高频 → 重复 → 得到有序 merge rules。**

你现在再看 GPT-2 的 `encoder.json` 和 `vocab.bpe`，就不会觉得它们是黑盒了。


### 加载一个真实 tokenizer 看看

下面用 `tiktoken` 加载 GPT-2 的真实 byte-level BPE tokenizer。

你先不用理解它所有工程细节，只看三件事：

1. 它真的会把文本切成 token；
2. 每个 token 都有一个固定 ID；
3. 空格、数字、换行、中文和 emoji 的切法，和我们的 mini 版不一样。


In [12]:
# 加载真实 GPT-2 tokenizer（byte-level BPE）
try:
    import tiktoken

    real_tokenizer_name = "gpt2"
    real_tokenizer = tiktoken.get_encoding(real_tokenizer_name)
    print(f"真实 tokenizer: {real_tokenizer_name}")
    print(f"词表大小: {real_tokenizer.n_vocab}")
except Exception as e:
    real_tokenizer = None
    print("未能加载 tiktoken 的 GPT-2 tokenizer")
    print(f"原因: {e}")
    print("安装并缓存 tiktoken 后，这个 cell 会打印真实 tokenizer 结果。")


def show_real_tokenization(text):
    """打印真实 tokenizer 如何把文本切成 token 和 ID"""
    ids = real_tokenizer.encode(text)
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]

    print(f"文本: {text!r}")
    print(f"token 数: {len(ids)}")
    for i, (tok_id, token) in enumerate(zip(ids, tokens)):
        visible = token.replace("\n", "\\n").replace("\t", "\\t")
        token_bytes = list(real_tokenizer.decode_single_token_bytes(tok_id))
        print(f"  {i:02d} | id={tok_id:<6} | token={visible!r} | bytes={token_bytes}")
    return ids, tokens


if real_tokenizer is not None:
    show_real_tokenization("the cat sat on the mat")


真实 tokenizer: gpt2
词表大小: 50257
文本: 'the cat sat on the mat'
token 数: 6
  00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
  02 | id=3332   | token=' sat' | bytes=[32, 115, 97, 116]
  03 | id=319    | token=' on' | bytes=[32, 111, 110]
  04 | id=262    | token=' the' | bytes=[32, 116, 104, 101]
  05 | id=2603   | token=' mat' | bytes=[32, 109, 97, 116]


### 和我们手写的 BPE 对比

我们的 mini BPE 只在 10 条英文小语料上训练，所以它的词表很小。
真实 tokenizer 在海量语料上训练，并且从 bytes 开始，所以能处理任意 Unicode 文本。


In [13]:
def try_mini_bpe(text):
    """用我们的 mini BPE 编码；失败时返回错误原因"""
    try:
        ids = bpe.encode(text)
        tokens = [bpe.itos[i] for i in ids]
        return ids, tokens, None
    except Exception as e:
        return None, None, e


compare_texts = [
    "the cat sat on the mat",
    " the cat",
    "the  cat",
    "327 + 1 = 328",
    "def f():\n    return x",
    "你好，世界🙂",
]

if real_tokenizer is not None:
    for text in compare_texts:
        print("=" * 68)
        print(f"文本: {text!r}")

        mini_ids, mini_tokens, error = try_mini_bpe(text)
        if error is None:
            print(f"mini BPE: {len(mini_ids)} tokens")
            print(f"  tokens: {mini_tokens}")
            print(f"  ids:    {mini_ids}")
        else:
            print("mini BPE: 无法编码")
            print(f"  原因: 小语料词表里没有 {error!r}")

        real_ids = real_tokenizer.encode(text)
        real_tokens = [real_tokenizer.decode([tok_id]) for tok_id in real_ids]
        real_tokens = [t.replace("\n", "\\n") for t in real_tokens]
        print(f"真实 GPT-2 BPE: {len(real_ids)} tokens")
        print(f"  tokens: {real_tokens}")
        print(f"  ids:    {real_ids}")

    print("=" * 68)
    print("关键观察：真实 tokenizer 不只是更大，还处理了 bytes、空格、换行和 Unicode。")
    print("我们的 mini BPE 用来理解 merge rules；工业版解决覆盖率、速度和边界情况。")


文本: 'the cat sat on the mat'
mini BPE: 6 tokens
  tokens: ['the cat ', 'sat o', 'n', ' the ', 'm', 'at']
  ids:    [31, 26, 17, 1, 16, 3]
真实 GPT-2 BPE: 6 tokens
  tokens: ['the', ' cat', ' sat', ' on', ' the', ' mat']
  ids:    [1169, 3797, 3332, 319, 262, 2603]
文本: ' the cat'
mini BPE: 3 tokens
  tokens: [' ', 'the c', 'at']
  ids:    [0, 30, 3]
真实 GPT-2 BPE: 2 tokens
  tokens: [' the', ' cat']
  ids:    [262, 3797]
文本: 'the  cat'
mini BPE: 4 tokens
  tokens: ['the ', ' ', 'c', 'at']
  ids:    [29, 0, 5, 3]
真实 GPT-2 BPE: 3 tokens
  tokens: ['the', ' ', ' cat']
  ids:    [1169, 220, 3797]
文本: '327 + 1 = 328'
mini BPE: 4 tokens
  tokens: [' ', ' ', ' ', ' ']
  ids:    [0, 0, 0, 0]
真实 GPT-2 BPE: 5 tokens
  tokens: ['327', ' +', ' 1', ' =', ' 328']
  ids:    [34159, 1343, 352, 796, 39093]
文本: 'def f():\n    return x'
mini BPE: 16 tokens
  tokens: ['d', 'e', 'f', ' ', 'f', ' ', ' ', ' ', ' ', 'r', 'e', 't', 'u', 'r', 'n', ' ']
  ids:    [6, 8, 10, 0, 10, 0, 0, 0, 0, 22, 8, 27, 32, 22, 17, 

### 特殊情况：数字、空格、缩进和 special token

真实 tokenizer 的很多“怪现象”，其实都来自训练语料和工程设计。
下面这些输出很重要：它们解释了为什么模型做数学、写代码、处理中英文混合时，
会受到 tokenizer 切法影响。


In [14]:
special_cases = [
    ("数字", "327 + 1 = 328"),
    ("词首空格", "cat"),
    ("同一个词，前面带空格", " cat"),
    ("多个空格", "the    cat"),
    ("换行和缩进", "def f():\n    return x"),
    ("中文和 emoji", "你好，世界🙂"),
]

if real_tokenizer is not None:
    for label, text in special_cases:
        print("=" * 68)
        print(f"特殊情况: {label}")
        show_real_tokenization(text)

    print("=" * 68)
    print("Special token: <|endoftext|>")
    text = "hello<|endoftext|>world"

    try:
        real_tokenizer.encode(text)
    except ValueError as e:
        print("默认 encode 会拒绝把 special token 当普通文本吞掉。")
        print(f"错误摘要: {str(e).splitlines()[0]}")

    ids = real_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
    print("允许 special token 后:")
    print(f"  ids:    {ids}")
    print(f"  tokens: {tokens}")
    print("关键：special token 是额外控制符，不是普通 merge rule 自己学出来的。")


特殊情况: 数字
文本: '327 + 1 = 328'
token 数: 5
  00 | id=34159  | token='327' | bytes=[51, 50, 55]
  01 | id=1343   | token=' +' | bytes=[32, 43]
  02 | id=352    | token=' 1' | bytes=[32, 49]
  03 | id=796    | token=' =' | bytes=[32, 61]
  04 | id=39093  | token=' 328' | bytes=[32, 51, 50, 56]
特殊情况: 词首空格
文本: 'cat'
token 数: 1
  00 | id=9246   | token='cat' | bytes=[99, 97, 116]
特殊情况: 同一个词，前面带空格
文本: ' cat'
token 数: 1
  00 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
特殊情况: 多个空格
文本: 'the    cat'
token 数: 5
  00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=220    | token=' ' | bytes=[32]
  02 | id=220    | token=' ' | bytes=[32]
  03 | id=220    | token=' ' | bytes=[32]
  04 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
特殊情况: 换行和缩进
文本: 'def f():\n    return x'
token 数: 9
  00 | id=4299   | token='def' | bytes=[100, 101, 102]
  01 | id=277    | token=' f' | bytes=[32, 102]
  02 | id=33529  | token='():' | bytes=[40, 41, 58]
  03 | id=198    | token='\\n' | bytes=[10]

### 趣味问题：为什么模型有时会觉得 `1.11` 比 `1.9` 大？

先别急着看答案，自己先猜一下：

```text
1.11 和 1.9，谁更大？
```

数学上当然是 `1.9` 更大。因为小数可以补 0：

```text
1.9  = 1.90
1.11 = 1.11
所以 1.90 > 1.11
```

但 LLM 不会一上来就把 `1.11` 当成“小数对象”。
它最先看到的是 tokenizer 切出来的 token ID。

如果切法长这样：

```text
"1.11" → ["1", ".", "11"]
"1.9"  → ["1", ".", "9"]
```

模型可能会被表面模式带偏：`11` 看起来比 `9` 大，于是误以为 `1.11 > 1.9`。

这不是说模型完全不会做小数比较，而是说：**它要先从 token 序列里学会“小数位对齐”这个规则。**
如果训练数据里这种例子不够多，或者问题问得很像字符串比较，它就容易犯错。

In [15]:
# 趣味观察：真实 tokenizer 怎么切 1.11 和 1.9
number_texts = ["1.11", "1.9"]

if real_tokenizer is not None:
    for text in number_texts:
        ids = real_tokenizer.encode(text)
        tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
        print(f"{text!r} -> tokens={tokens}, ids={ids}")

    print()
    print("数学比较：")
    print(f"  1.11 > 1.9  ? {1.11 > 1.9}")
    print(f"  1.90 > 1.11 ? {1.90 > 1.11}")
    print()
    print("关键观察：模型看到的不是 float，而是 token 序列。")
    print("要比较小数，它必须学会把小数位补齐，而不是直接比较 token '11' 和 '9'。")

'1.11' -> tokens=['1', '.', '11'], ids=[16, 13, 1157]
'1.9' -> tokens=['1', '.', '9'], ids=[16, 13, 24]

数学比较：
  1.11 > 1.9  ? False
  1.90 > 1.11 ? True

关键观察：模型看到的不是 float，而是 token 序列。
要比较小数，它必须学会把小数位补齐，而不是直接比较 token '11' 和 '9'。


**Keypoint 10：Special Tokens 不属于 BPE merge rules**

这里先记住边界就够了：

```text
BPE 学出来的 token:      th + e → the
人手动约定的控制符号:     <BOS>、<EOS>、<PAD>、<think>、</think>
```

BPE 负责把普通文本切成 token。Special tokens 负责告诉模型“哪里开始、哪里结束、哪里是 padding、哪里是 thinking 区间”。

所以它们通常不是靠统计 pair 学出来的，而是在训练 tokenizer 或扩展 tokenizer 时额外加入词表。
正式怎么用，会放到 Mini-GPT 和 Thinking 模型里讲。

**可视化：merge rules 怎样一步步长出来？**

最后看一个小彩蛋：把 merge rules 画出来。

这张图不是为了炫技，而是帮你把前面的直觉固定住：常见组合会先出现，然后更长的组合在它们上面继续生长。


In [16]:
print("Merge Rules 完整列表（按训练顺序）:")
print("注意看 'the' 是怎么诞生的：")
print()

for i, (a, b) in enumerate(bpe.merge_rules):
    arrow = ""
    # 标记和 'the' 相关的合并
    merged = a + b
    if merged in ['th', 'the ', 'the c', 'the cat ']:
        arrow = "  ← 'the' 的诞生过程！"
    if a in ['th', 'the ', 'the c'] or b in ['e ', 'c', 'at ']:
        arrow = "  ← 'the' 的诞生过程！"
    print(f"  Rule {i+1:2d}: '{a}' + '{b}' → '{merged}'{arrow}")

print()
print("你看：'the ' 不是一次性识别出来的，是 't'+'h'→'th'，再和 'e ' 合并，逐渐形成的。")

Merge Rules 完整列表（按训练顺序）:
注意看 'the' 是怎么诞生的：

  Rule  1: 'e' + ' ' → 'e '
  Rule  2: 't' + 'h' → 'th'  ← 'the' 的诞生过程！
  Rule  3: 'th' + 'e ' → 'the '  ← 'the' 的诞生过程！
  Rule  4: 'a' + 't' → 'at'
  Rule  5: 'o' + 'g' → 'og'
  Rule  6: 'at' + ' ' → 'at '
  Rule  7: 's' + ' ' → 's '
  Rule  8: 'd' + 'og' → 'dog'
  Rule  9: 'i' + 's ' → 'is '
  Rule 10: 'the ' + 'c' → 'the c'  ← 'the' 的诞生过程！
  Rule 11: 'the c' + 'at ' → 'the cat '  ← 'the' 的诞生过程！
  Rule 12: ' ' + 'the ' → ' the '
  Rule 13: 'n' + 'd' → 'nd'
  Rule 14: 's' + 'at ' → 'sat '  ← 'the' 的诞生过程！
  Rule 15: 'sat ' + 'o' → 'sat o'

你看：'the ' 不是一次性识别出来的，是 't'+'h'→'th'，再和 'e ' 合并，逐渐形成的。


---

## 小结

对照检查一下：

1. ✅ BPE 从字符或 byte 开始，因为起点要稳
2. ✅ 每一步都统计相邻 pair，合并最高频的那个
3. ✅ merge rules 是有顺序的，encode 新文本时要按顺序重放
4. ✅ `num_merges` 控制词表大小，也控制 token 切得有多“整”
5. ✅ decode 很简单：查表，然后把字符串拼回去
6. ✅ BPE 不怕大多数生词，因为它可以退回到更小的子词或字符
7. ✅ 数字也是 token 序列，模型要学会小数位对齐，不能只看表面字符串

如果你只记一个画面：**BPE 就是让 token 从字符开始，一层一层长成常见子词。**

下一步 → **Part 3**：token ID 只是编号，神经网络不能直接理解编号。那怎么把编号变成向量？我们进入 Embedding + Position Encoding。


---

## 本章作业：BPE Tokenizer

这 3 题分成两类：

1. **核心必须记住**：BPE 反复统计相邻 pair，并合并最高频 pair。
2. **现代 LLM 怎么用**：真实 tokenizer 常从 bytes 开始，并按有序 merge rules 编码。

> **使用 AI 的注意事项**：可以让 AI 提示“下一步怎么合并”，但不要让它直接写完。BPE 的重点是你亲眼看到 token 怎么一步步长出来。

### 作业 1：核心机制 —— 统计 pair

BPE 的第一步是统计相邻 token pair 的频率。

**小提示**：pair 是 `(tokens[i], tokens[i + 1])`。

In [17]:
# 作业 1：统计相邻 pair 填空
from collections import defaultdict

tokens = list("banana")
pair_counts = defaultdict(int)

# TODO：把下面三引号里的内容替换成你的代码
"""在这里统计所有相邻 pair"""

assert pair_counts[("a", "n")] == 2, dict(pair_counts)
assert pair_counts[("n", "a")] == 2, dict(pair_counts)
assert pair_counts[("b", "a")] == 1, dict(pair_counts)
print("✅ 作业 1 通过：你记住了 BPE 的第一步是统计相邻 pair")

AssertionError: {('a', 'n'): 0}

### 作业 2：核心机制 —— 合并最高频 pair

找到高频 pair 后，要把它们合并成一个新 token。

**小提示**：命中 pair 时，一次消耗两个 token；没命中时，只消耗当前 token。

In [ ]:
# 作业 2：合并 pair 填空
def merge_pair(tokens, pair):
    """把 tokens 中等于 pair 的相邻 token 合并成一个新 token"""
    merged = []
    i = 0
    while i < len(tokens):
        # TODO：把下面三引号里的内容替换成你的代码
        """在这里判断是否命中 pair，并更新 merged 和 i"""
    return merged

result = merge_pair(list("banana"), ("a", "n"))

assert result == ["b", "an", "an", "a"], result
print("✅ 作业 2 通过：你记住了 BPE 的核心动作是 merge")

### 作业 3：现代用法 —— byte 级起点

现代 tokenizer 常从 bytes 开始，这样任何 Unicode 字符都能先进门。

**小提示**：Python 里可以用 `text.encode("utf-8")` 查看一个字符串对应的 bytes。

In [ ]:
# 作业 3：UTF-8 bytes 填空
text = "你😊"

# TODO：把下面三引号里的内容替换成你的代码
byte_values = """在这里把 text 编码成 UTF-8 byte 数值列表"""

assert not isinstance(byte_values, str), "请先替换三引号里的占位内容"
assert byte_values == [228, 189, 160, 240, 159, 152, 138], byte_values
print("✅ 作业 3 通过：你理解了为什么现代 tokenizer 常从 bytes 开始")